# ✈️ Airspeed Protection Reflex Agent
### LangGraph + Groq (LLaMA 3.1 8B) + Gradio UI

Prevents aircraft stall using a simple reflex agent system.

In [ ]:
# 1. Install necessary libraries
!pip install langgraph langchain langchain-core langchain-community groq gradio

In [ ]:
# 2. Import libraries
import os
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_community.chat_models import ChatGroq

import gradio as gr

In [ ]:
# 3. Set Groq API Key
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

## 4. Reflex Logic

In [ ]:
# Define system state
class FlightState(TypedDict):
    airspeed: float
    aoa: float
    altitude: float
    decision: str


# Thresholds
STALL_SPEED_THRESHOLD = 120  # knots
CRITICAL_AOA = 15           # degrees


def reflex_logic(state: FlightState):
    airspeed = state["airspeed"]
    aoa = state["aoa"]

    if airspeed < STALL_SPEED_THRESHOLD:
        decision = "⚠️ Stall Risk: Pitch Nose Down Immediately"
    elif aoa > CRITICAL_AOA:
        decision = "⚠️ High AoA: Reduce Pitch"
    else:
        decision = "✅ Stable Flight: Maintain Current Control"

    state["decision"] = decision
    return state

## 5. LLM Decision Layer

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


def llm_decision(state: FlightState):
    prompt = f"""
    You are an aircraft safety system.

    Sensor Readings:
    - Airspeed: {state['airspeed']} knots
    - Angle of Attack: {state['aoa']} degrees
    - Altitude: {state['altitude']} ft

    Rule-based Decision:
    {state['decision']}

    Convert this into a clear professional pilot instruction.
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    state["decision"] = response.content
    return state

## 6. Build LangGraph Workflow

In [ ]:
builder = StateGraph(FlightState)

builder.add_node("reflex", reflex_logic)
builder.add_node("llm_refine", llm_decision)

builder.set_entry_point("reflex")
builder.add_edge("reflex", "llm_refine")
builder.add_edge("llm_refine", END)

graph = builder.compile()

## 7. Gradio UI

In [ ]:
def run_agent(airspeed, aoa, altitude):
    state = {
        "airspeed": airspeed,
        "aoa": aoa,
        "altitude": altitude,
        "decision": ""
    }

    result = graph.invoke(state)
    return result["decision"]


with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    <h1 style="text-align:center; color:#2c3e50;">
    ✈️ Airspeed Protection Reflex Agent
    </h1>

    <marquee behavior="scroll" direction="left" style="color:#3498db; font-size:18px;">
    Airspeed Protection Reflex Agent — Real-Time Aircraft Safety System
    </marquee>
    """)

    with gr.Row():
        airspeed = gr.Slider(50, 300, value=130, label="Airspeed (knots)")
        aoa = gr.Slider(0, 25, value=10, label="Angle of Attack (°)")
        altitude = gr.Slider(0, 40000, value=10000, label="Altitude (ft)")

    output = gr.Textbox(label="Control Action", lines=4)

    btn = gr.Button("Evaluate Flight Safety")

    btn.click(fn=run_agent, inputs=[airspeed, aoa, altitude], outputs=output)

app.launch()